# RSI-Plateau: Free-T4 Validation (Real Model Path)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbhiGuru25/Recursive-self-improvement-RSI/blob/main/notebooks/t4_validation.ipynb)

**Goal:** Prove the *real* pipeline works on a GPU before spending budget on the Tier-1
matrix. Runs a tiny STaR loop with Qwen2.5-0.5B on a small GSM8K subset, oracle verifier,
LoRA SFT, 2 rounds.

### Before you run
1. `Runtime` -> `Change runtime type` -> **T4 GPU** -> Save.
2. `Runtime` -> `Run all`.

**Expected:** non-zero accuracy, round-to-round change, a `result.json`. These are
**plumbing numbers, not paper numbers** (PRD section 5.9, Tier 0). ~10-20 min.

> Your repo is public, so cloning needs no token. If you make it private, use
> `git clone https://<TOKEN>@github.com/AbhiGuru25/Recursive-self-improvement-RSI.git`.

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Clone the repo and install (Colab already ships torch/transformers)
import os
REPO = "https://github.com/AbhiGuru25/Recursive-self-improvement-RSI.git"
if not os.path.isdir("RSI"):
    !git clone -q $REPO RSI
%cd RSI
!pip install -q -e ".[tier0]"

In [ ]:
# 3. Sanity: torch sees the GPU, and pick a dtype the T4 supports
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."
print("cuda:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())
print("free VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# 4. Write a GPU smoke config (0.5B, small subset, auto precision)
import pathlib, yaml

cfg = {
    "run": {"name": "t4_smoke", "seed": 0, "rounds": 2,
            "output_dir": "artifacts/t4_smoke", "use_wandb": False},
    "data": {"dataset": "gsm8k", "train_size": 50, "test_size": 100},
    "model": {"policy": "Qwen/Qwen2.5-0.5B-Instruct", "dtype": "float16",
              "device": "cuda", "judge": None},
    "generation": {"k_samples": 2, "temperature": 0.8, "top_p": 0.95,
                   "max_new_tokens": 256, "batch_size": 8},
    "verifier": {"type": "oracle", "match_acceptance_rate": None, "calibration_size": 0},
    "loop": {"architecture": "star", "frozen_reference_diversity": True},
    "training": {"method": "lora", "lora_r": 8, "lora_alpha": 16, "lora_dropout": 0.05,
                 "learning_rate": 0.0001, "epochs": 1, "batch_size": 2, "grad_accum": 4,
                 "max_seq_len": 512, "precision": "auto"},
    "diversity": {"output_entropy": True, "embedding_clustering": False, "self_bleu": True},
    "stats": {"bootstrap_samples": 200, "ci_alpha": 0.05,
              "plateau_abs_delta": 0.005, "plateau_consecutive": 2},
}
pathlib.Path("configs").mkdir(exist_ok=True)
with open("configs/t4_smoke.yaml", "w") as fh:
    yaml.safe_dump(cfg, fh)
print(open("configs/t4_smoke.yaml").read())

In [ ]:
# 5. Run the real loop (HF generation + LoRA SFT) on the T4
!python scripts/run_loop.py --config configs/t4_smoke.yaml 2>&1 | tail -40

In [ ]:
# 6. Inspect the result artifact
import json
d = json.load(open("artifacts/t4_smoke/result.json"))
print("env:", d["metadata"].get("gpu"))
for r in d["result"]["rounds"]:
    print(r["round"], "acc=", round(r["accuracy"], 3),
          "kept=", f"{r['n_train_kept']}/{r['n_train_total']}",
          "self_bleu=", round(r["diversity"].get("self_bleu", -1), 3),
          "train_loss=", r.get("train_loss"))
print("plateau:", d["result"]["plateau"])

## Interpreting the result

Success = the loop ran end-to-end on GPU: sampling worked, oracle filtering kept some
samples, LoRA training completed, evaluation produced a number, and `result.json` exists.

Accuracy will be low (0.5B model, 2 rounds, 50 training questions) — expected and fine.
The point is to de-risk the *backends* (HF generate, PEFT/TRL SFT, dtype/device) before
launching the paid Tier-1 matrix. If this passes, the remaining risk is budget, not code.

### If something fails
- **CUDA out of memory:** lower `batch_size` to 1 and `max_new_tokens` to 128 in cell 4.
- **Zero kept samples:** the 0.5B model is weak; that is normal. Try a larger model
  (`Qwen/Qwen2.5-1.5B-Instruct`) if you have Colab Pro, or raise `k_samples`.